In [1]:
from pathlib import Path
import pandas as pd

In [3]:
BASE = Path("data/processed_full/enrichment")
OUT = Path("data/processed_full/manual_fusions")
OUT.mkdir(parents=True, exist_ok=True)

main_path = BASE / "main_associations_enriched.csv"
gene_links_path = BASE / "association_pm_gene_links_top.csv"
phenotype_links_path = BASE / "association_pm_phenotype_links.csv"
disease_mapping_path = BASE / "ctd_pm_disease_mapping.csv"
phenotype_mapping_path = BASE / "ctd_pm_phenotype_mapping.csv"
gene_top_path = BASE / "ctd_pm_human_genes_top.csv"

In [3]:
main = pd.read_csv(main_path, low_memory=False)
gene_links = pd.read_csv(gene_links_path, low_memory=False)
phenotype_links = pd.read_csv(phenotype_links_path, low_memory=False)
disease_mapping = pd.read_csv(disease_mapping_path, low_memory=False)
phenotype_mapping = pd.read_csv(phenotype_mapping_path, low_memory=False)
gene_top = pd.read_csv(gene_top_path, low_memory=False)

print("main:", main.shape)
print("gene_links:", gene_links.shape)
print("phenotype_links:", phenotype_links.shape)
print("disease_mapping:", disease_mapping.shape)
print("phenotype_mapping:", phenotype_mapping.shape)
print("gene_top:", gene_top.shape)

main: (2368, 70)
gene_links: (366400, 22)
phenotype_links: (58, 20)
disease_mapping: (562, 17)
phenotype_mapping: (562, 20)
gene_top: (200, 13)


In [4]:
print("main association_id unique:", main["association_id"].is_unique)
print("main rows:", len(main))
print("main unique association_id:", main["association_id"].nunique())

print("gene_links rows:", len(gene_links))
print("gene_links unique association_id:", gene_links["association_id"].nunique())

print("phenotype_links rows:", len(phenotype_links))
print("phenotype_links unique association_id:", phenotype_links["association_id"].nunique())

if "outcome_key" in main.columns:
    print("main outcome_key unique:", main["outcome_key"].nunique())

if "outcome_key" in disease_mapping.columns:
    print("disease_mapping outcome_key unique:", disease_mapping["outcome_key"].is_unique)

if "outcome_key" in phenotype_mapping.columns:
    print("phenotype_mapping outcome_key unique:", phenotype_mapping["outcome_key"].is_unique)

main association_id unique: True
main rows: 2368
main unique association_id: 2368
gene_links rows: 366400
gene_links unique association_id: 1832
phenotype_links rows: 58
phenotype_links unique association_id: 58
main outcome_key unique: 675
disease_mapping outcome_key unique: True
phenotype_mapping outcome_key unique: True


In [5]:
gene_agg = (
    gene_links
    .groupby("association_id")
    .agg(
        gene_count=("gene_symbol", "nunique"),
        gene_symbols=("gene_symbol", lambda x: ";".join(sorted(set(map(str, x))))),
        gene_ids=("gene_id", lambda x: ";".join(sorted(set(map(str, x))))),
        gene_actions=("interaction_actions_top", lambda x: ";".join(sorted(set(map(str, x.dropna()))))),
        gene_reference_count_sum=("reference_count_from_ixns", "sum"),
    )
    .reset_index()
)

print(gene_agg.shape)
print(gene_agg.head())

(1832, 6)
                association_id  gene_count  \
0  assoc_12842775_fe9cb68c6a56         200   
1  assoc_14998750_44ce04994750         200   
2  assoc_14998750_49915f02c18a         200   
3  assoc_14998750_53fe372da300         200   
4  assoc_14998750_7c8fc6671cfd         200   

                                        gene_symbols  \
0  ADRB2;AHR;AKR1C1;AKR1C3;AKT1;ALB;ALDH1A3;ALDH3...   
1  ADRB2;AHR;AKR1C1;AKR1C3;AKT1;ALB;ALDH1A3;ALDH3...   
2  ADRB2;AHR;AKR1C1;AKR1C3;AKT1;ALB;ALDH1A3;ALDH3...   
3  ADRB2;AHR;AKR1C1;AKR1C3;AKT1;ALB;ALDH1A3;ALDH3...   
4  ADRB2;AHR;AKR1C1;AKR1C3;AKT1;ALB;ALDH1A3;ALDH3...   

                                            gene_ids  \
0  1000;100506658;102466659;1026;1029;10397;1052;...   
1  1000;100506658;102466659;1026;1029;10397;1052;...   
2  1000;100506658;102466659;1026;1029;10397;1052;...   
3  1000;100506658;102466659;1026;1029;10397;1052;...   
4  1000;100506658;102466659;1026;1029;10397;1052;...   

                                       

In [6]:
phenotype_agg = (
    phenotype_links
    .groupby("association_id")
    .agg(
        phenotype_count=("ctd_phenotype_name", "nunique"),
        phenotype_names=("ctd_phenotype_name", lambda x: ";".join(sorted(set(map(str, x))))),
        phenotype_ids=("ctd_phenotype_id", lambda x: ";".join(sorted(set(map(str, x))))),
        phenotype_interactions=("ctd_phenotype_interaction", lambda x: ";".join(sorted(set(map(str, x.dropna()))))),
    )
    .reset_index()
)

print(phenotype_agg.shape)
print(phenotype_agg.head())

(58, 5)
                association_id  phenotype_count      phenotype_names  \
0  assoc_20123602_752ef613c984                1  platelet activation   
1  assoc_21169126_0c2927054ce6                1          methylation   
2  assoc_21169126_6e4fe6869b78                1          methylation   
3  assoc_21169126_a452c2607972                1          methylation   
4  assoc_21169126_e83bfa71abcd                1          methylation   

  phenotype_ids                             phenotype_interactions  
0    GO:0030168  Particulate Matter results in increased platel...  
1    GO:0032259  Particulate Matter results in decreased methyl...  
2    GO:0032259  Particulate Matter results in decreased methyl...  
3    GO:0032259  Particulate Matter results in decreased methyl...  
4    GO:0032259  Particulate Matter results in decreased methyl...  


In [7]:
main_flat = (
    main
    .merge(gene_agg, on="association_id", how="left", validate="1:1")
    .merge(phenotype_agg, on="association_id", how="left", validate="1:1")
)

# Remplir les valeurs manquantes
for col in ["gene_count", "phenotype_count"]:
    if col in main_flat.columns:
        main_flat[col] = main_flat[col].fillna(0).astype(int)

print("main:", main.shape)
print("main_flat:", main_flat.shape)

assert len(main_flat) == len(main), "Erreur: la fusion a changé le nombre de lignes."

main_flat.to_csv(OUT / "main_flat_with_genes_phenotypes.csv", index=False)

main_flat.head()

main: (2368, 70)
main_flat: (2368, 79)


,association_id,pmid,association_label,association_claim,pollutant_outcome_key,pollutant_general,pollutant_specific,pollutant_original_mention,outcome_normalized,outcome_core,...,mechanism_available,gene_count,gene_symbols,gene_ids,gene_actions,gene_reference_count_sum,phenotype_count,phenotype_names,phenotype_ids,phenotype_interactions
0,assoc_12842775_fe9cb68c6a56,12842775,PM10 -> daily mortality,PM10 is associated with increased daily mortality,pm10__daily_mortality,PM,PM10,PM10,daily mortality,daily mortality,...,True,200,ADRB2;AHR;AKR1C1;AKR1C3;AKT1;ALB;ALDH1A3;ALDH3...,1000;100506658;102466659;1026;1029;10397;1052;...,affects^cotreatment;decreases^expression;incre...,2716.0,0,NaN,NaN,NaN
1,assoc_12882940_64b5dfa384e6,12882940,NO2 -> persistent cough,NO2 is associated with increased persistent cough,no2__persistent_cough,NO2,NO2,nitrogen dioxide,persistent cough,persistent cough,...,False,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
2,assoc_14998750_49915f02c18a,14998750,PM2.5 -> decline in standard deviation of all ...,PM2.5 is associated with increased decline in ...,pm2_5__decline_in_standard_deviation_of_all_no...,PM,PM2.5,fine particulate matter (PM2.5),decline in standard deviation of all normal R-...,decline in standard deviation of all normal r ...,...,True,200,ADRB2;AHR;AKR1C1;AKR1C3;AKT1;ALB;ALDH1A3;ALDH3...,1000;100506658;102466659;1026;1029;10397;1052;...,affects^cotreatment;decreases^expression;incre...,2716.0,0,NaN,NaN,NaN
3,assoc_14998750_7c8fc6671cfd,14998750,PM2.5 -> decline in short-term heart rate vari...,PM2.5 is associated with increased decline in ...,pm2_5__decline_in_short_term_heart_rate_variab...,PM,PM2.5,fine particulate matter (PM2.5),decline in short-term heart rate variability,decline in short term heart rate variability,...,True,200,ADRB2;AHR;AKR1C1;AKR1C3;AKT1;ALB;ALDH1A3;ALDH3...,1000;100506658;102466659;1026;1029;10397;1052;...,affects^cotreatment;decreases^expression;incre...,2716.0,0,NaN,NaN,NaN
4,assoc_14998750_44ce04994750,14998750,PM2.5 -> C-reactive protein (CRP) increase,PM2.5 is associated with increased C-reactive ...,pm2_5__c_reactive_protein_crp_increase,PM,PM2.5,fine particulate matter (PM2.5),C-reactive protein (CRP) increase,c reactive protein (crp) increase,...,True,200,ADRB2;AHR;AKR1C1;AKR1C3;AKT1;ALB;ALDH1A3;ALDH3...,1000;100506658;102466659;1026;1029;10397;1052;...,affects^cotreatment;decreases^expression;incre...,2716.0,0,NaN,NaN,NaN


In [8]:
main_gene_long = main.merge(
    gene_links,
    on="association_id",
    how="inner",
    suffixes=("_assoc", "_gene"),
    validate="1:m"
)

print("main:", main.shape)
print("gene_links:", gene_links.shape)
print("main_gene_long:", main_gene_long.shape)

main_gene_long.to_csv(OUT / "main_gene_long.csv", index=False)

main_gene_long.head()

main: (2368, 70)
gene_links: (366400, 22)
main_gene_long: (366400, 91)


,association_id,pmid_assoc,association_label,association_claim,pollutant_outcome_key,pollutant_general_assoc,pollutant_specific_assoc,pollutant_original_mention,outcome_normalized_assoc,outcome_core_assoc,...,ctd_chemical_name_gene,human_ixn_row_count,interaction_actions_top,reference_count_from_ixns,references_sample,organisms,species_filter_status,ctd_gene_interaction_count_global,ctd_gene_organism_count_global,gene_rank_pm_human_ixns
0,assoc_12842775_fe9cb68c6a56,12842775,PM10 -> daily mortality,PM10 is associated with increased daily mortality,pm10__daily_mortality,PM,PM10,PM10,daily mortality,daily mortality,...,Particulate Matter,124,increases^expression;decreases^reaction;increa...,103,29227181;10919993;16076765;27295300;40345318;2...,Homo sapiens;Mus musculus,human_only,398,5,1
1,assoc_12842775_fe9cb68c6a56,12842775,PM10 -> daily mortality,PM10 is associated with increased daily mortality,pm10__daily_mortality,PM,PM10,PM10,daily mortality,daily mortality,...,Particulate Matter,100,increases^expression;decreases^reaction;increa...,177,40348349;27295300;29753051;29979839;40199452;2...,Homo sapiens;Mus musculus;Rattus norvegicus,human_only,677,5,2
2,assoc_12842775_fe9cb68c6a56,12842775,PM10 -> daily mortality,PM10 is associated with increased daily mortality,pm10__daily_mortality,PM,PM10,PM10,daily mortality,daily mortality,...,Particulate Matter,83,decreases^reaction;increases^expression;increa...,159,15613243;40348349;28655636;29753051;39189708;4...,Homo sapiens;Rattus norvegicus;Mus musculus;Or...,human_only,562,6,3
3,assoc_12842775_fe9cb68c6a56,12842775,PM10 -> daily mortality,PM10 is associated with increased daily mortality,pm10__daily_mortality,PM,PM10,PM10,daily mortality,daily mortality,...,Particulate Matter,79,increases^expression;decreases^reaction;increa...,126,40348349;40199452;32922182;32687961;22867088;3...,Homo sapiens;Rattus norvegicus;Mus musculus;Or...,human_only,489,6,4
4,assoc_12842775_fe9cb68c6a56,12842775,PM10 -> daily mortality,PM10 is associated with increased daily mortality,pm10__daily_mortality,PM,PM10,PM10,daily mortality,daily mortality,...,Particulate Matter,58,decreases^reaction;increases^expression;increa...,42,30354488;31822346;29979839;39189708;28867211;2...,Homo sapiens;Mus musculus;Rattus norvegicus;Su...,human_only,202,4,5


In [9]:
# Sécuriser les clés comme string
gene_links["gene_symbol"] = gene_links["gene_symbol"].astype(str)
gene_links["gene_id"] = gene_links["gene_id"].astype(str)

gene_top["gene_symbol"] = gene_top["gene_symbol"].astype(str)
gene_top["gene_id"] = gene_top["gene_id"].astype(str)

gene_links_with_meta = gene_links.merge(
    gene_top,
    on=["gene_symbol", "gene_id"],
    how="left",
    suffixes=("", "_gene_meta"),
    validate="m:1"
)

print("gene_links:", gene_links.shape)
print("gene_links_with_meta:", gene_links_with_meta.shape)

gene_links_with_meta.to_csv(OUT / "gene_links_with_metadata.csv", index=False)

gene_links_with_meta.head()

gene_links: (366400, 22)
gene_links_with_meta: (366400, 33)


,association_id,pmid,pollutant_specific,pollutant_general,outcome_normalized,outcome_core,measure_type,effect_value,effect_direction,gene_symbol,...,ctd_chemical_name_gene_meta,human_ixn_row_count_gene_meta,interaction_actions_top_gene_meta,reference_count_from_ixns_gene_meta,references_sample_gene_meta,organisms_gene_meta,species_filter_status_gene_meta,ctd_gene_interaction_count_global_gene_meta,ctd_gene_organism_count_global_gene_meta,gene_rank_pm_human_ixns_gene_meta
0,assoc_12842775_fe9cb68c6a56,12842775,PM10,PM,daily mortality,daily mortality,percent_change,0.5,increase,CXCL8,...,Particulate Matter,124,increases^expression;decreases^reaction;increa...,103,29227181;10919993;16076765;27295300;40345318;2...,Homo sapiens;Mus musculus,human_only,398,5,1
1,assoc_12842775_fe9cb68c6a56,12842775,PM10,PM,daily mortality,daily mortality,percent_change,0.5,increase,IL6,...,Particulate Matter,100,increases^expression;decreases^reaction;increa...,177,40348349;27295300;29753051;29979839;40199452;2...,Homo sapiens;Mus musculus;Rattus norvegicus,human_only,677,5,2
2,assoc_12842775_fe9cb68c6a56,12842775,PM10,PM,daily mortality,daily mortality,percent_change,0.5,increase,TNF,...,Particulate Matter,83,decreases^reaction;increases^expression;increa...,159,15613243;40348349;28655636;29753051;39189708;4...,Homo sapiens;Rattus norvegicus;Mus musculus;Or...,human_only,562,6,3
3,assoc_12842775_fe9cb68c6a56,12842775,PM10,PM,daily mortality,daily mortality,percent_change,0.5,increase,IL1B,...,Particulate Matter,79,increases^expression;decreases^reaction;increa...,126,40348349;40199452;32922182;32687961;22867088;3...,Homo sapiens;Rattus norvegicus;Mus musculus;Or...,human_only,489,6,4
4,assoc_12842775_fe9cb68c6a56,12842775,PM10,PM,daily mortality,daily mortality,percent_change,0.5,increase,RELA,...,Particulate Matter,58,decreases^reaction;increases^expression;increa...,42,30354488;31822346;29979839;39189708;28867211;2...,Homo sapiens;Mus musculus;Rattus norvegicus;Su...,human_only,202,4,5


In [10]:
main_gene_meta_long = main.merge(
    gene_links_with_meta,
    on="association_id",
    how="inner",
    suffixes=("_assoc", "_gene"),
    validate="1:m"
)

print("main_gene_meta_long:", main_gene_meta_long.shape)

main_gene_meta_long.to_csv(OUT / "main_gene_meta_long.csv", index=False)

main_gene_meta_long.head()

main_gene_meta_long: (366400, 102)


,association_id,pmid_assoc,association_label,association_claim,pollutant_outcome_key,pollutant_general_assoc,pollutant_specific_assoc,pollutant_original_mention,outcome_normalized_assoc,outcome_core_assoc,...,ctd_chemical_name_gene_meta,human_ixn_row_count_gene_meta,interaction_actions_top_gene_meta,reference_count_from_ixns_gene_meta,references_sample_gene_meta,organisms_gene_meta,species_filter_status_gene_meta,ctd_gene_interaction_count_global_gene_meta,ctd_gene_organism_count_global_gene_meta,gene_rank_pm_human_ixns_gene_meta
0,assoc_12842775_fe9cb68c6a56,12842775,PM10 -> daily mortality,PM10 is associated with increased daily mortality,pm10__daily_mortality,PM,PM10,PM10,daily mortality,daily mortality,...,Particulate Matter,124,increases^expression;decreases^reaction;increa...,103,29227181;10919993;16076765;27295300;40345318;2...,Homo sapiens;Mus musculus,human_only,398,5,1
1,assoc_12842775_fe9cb68c6a56,12842775,PM10 -> daily mortality,PM10 is associated with increased daily mortality,pm10__daily_mortality,PM,PM10,PM10,daily mortality,daily mortality,...,Particulate Matter,100,increases^expression;decreases^reaction;increa...,177,40348349;27295300;29753051;29979839;40199452;2...,Homo sapiens;Mus musculus;Rattus norvegicus,human_only,677,5,2
2,assoc_12842775_fe9cb68c6a56,12842775,PM10 -> daily mortality,PM10 is associated with increased daily mortality,pm10__daily_mortality,PM,PM10,PM10,daily mortality,daily mortality,...,Particulate Matter,83,decreases^reaction;increases^expression;increa...,159,15613243;40348349;28655636;29753051;39189708;4...,Homo sapiens;Rattus norvegicus;Mus musculus;Or...,human_only,562,6,3
3,assoc_12842775_fe9cb68c6a56,12842775,PM10 -> daily mortality,PM10 is associated with increased daily mortality,pm10__daily_mortality,PM,PM10,PM10,daily mortality,daily mortality,...,Particulate Matter,79,increases^expression;decreases^reaction;increa...,126,40348349;40199452;32922182;32687961;22867088;3...,Homo sapiens;Rattus norvegicus;Mus musculus;Or...,human_only,489,6,4
4,assoc_12842775_fe9cb68c6a56,12842775,PM10 -> daily mortality,PM10 is associated with increased daily mortality,pm10__daily_mortality,PM,PM10,PM10,daily mortality,daily mortality,...,Particulate Matter,58,decreases^reaction;increases^expression;increa...,42,30354488;31822346;29979839;39189708;28867211;2...,Homo sapiens;Mus musculus;Rattus norvegicus;Su...,human_only,202,4,5


In [11]:
main_phenotype_long = main.merge(
    phenotype_links,
    on="association_id",
    how="inner",
    suffixes=("_assoc", "_phenotype"),
    validate="1:m"
)

print("main:", main.shape)
print("phenotype_links:", phenotype_links.shape)
print("main_phenotype_long:", main_phenotype_long.shape)

main_phenotype_long.to_csv(OUT / "main_phenotype_long.csv", index=False)

main_phenotype_long.head()

main: (2368, 70)
phenotype_links: (58, 20)
main_phenotype_long: (58, 89)


,association_id,pmid_assoc,association_label,association_claim,pollutant_outcome_key,pollutant_general_assoc,pollutant_specific_assoc,pollutant_original_mention,outcome_normalized_assoc,outcome_core_assoc,...,ctd_phenotype_id_phenotype,ctd_phenotype_interaction_phenotype,ctd_phenotype_organisms,ctd_phenotype_anatomy,ctd_phenotype_inference_network,ctd_phenotype_reference_count_phenotype,ctd_phenotype_references,ctd_phenotype_match_type_phenotype,ctd_phenotype_match_confidence_phenotype,ctd_phenotype_species_filter_status_phenotype
0,assoc_20123602_752ef613c984,20123602,PM10 -> platelet activation (PFA-100 closure t...,PM10 is associated with increased platelet act...,pm10__platelet_activation_pfa_100_closure_time...,PM,PM10,PM10,platelet activation (PFA-100 closure time decr...,platelet activation (pfa 100 closure time decr...,...,GO:0030168,Particulate Matter results in increased platel...,Homo sapiens,Blood Platelets,ACTN1|ADAMTS13|ADRA2B|ADRA2C|AXL|BLOC1S3|C1GAL...,1,20123602,candidate_contained_in_query,0.9,human_only
1,assoc_21169126_0c2927054ce6,21169126,PM10 -> hTERT methylation (position 3),PM10 is associated with decreased hTERT methyl...,pm10__htert_methylation_position_3,PM,PM10,PM10,hTERT methylation (position 3),htert methylation (position 3),...,GO:0032259,Particulate Matter results in decreased methyl...,Homo sapiens,Cells,ADGRV1|AS3MT|ATG13|BIN3|CCDC116|CMTR2|COMT|COQ...,1,22055874,candidate_contained_in_query,0.9,human_only
2,assoc_21169126_6e4fe6869b78,21169126,PM1 -> hTERT methylation (position 3),PM1 is associated with decreased hTERT methyla...,pm1__htert_methylation_position_3,PM,PM1,PM1,hTERT methylation (position 3),htert methylation (position 3),...,GO:0032259,Particulate Matter results in decreased methyl...,Homo sapiens,Cells,ADGRV1|AS3MT|ATG13|BIN3|CCDC116|CMTR2|COMT|COQ...,1,22055874,candidate_contained_in_query,0.9,human_only
3,assoc_21169126_e83bfa71abcd,21169126,PM10 -> hTERT methylation (position 3),PM10 is associated with decreased hTERT methyl...,pm10__htert_methylation_position_3,PM,PM10,PM10,hTERT methylation (position 3),htert methylation (position 3),...,GO:0032259,Particulate Matter results in decreased methyl...,Homo sapiens,Cells,ADGRV1|AS3MT|ATG13|BIN3|CCDC116|CMTR2|COMT|COQ...,1,22055874,candidate_contained_in_query,0.9,human_only
4,assoc_21169126_a452c2607972,21169126,PM1 -> hTERT methylation (position 3),PM1 is associated with decreased hTERT methyla...,pm1__htert_methylation_position_3,PM,PM1,PM1,hTERT methylation (position 3),htert methylation (position 3),...,GO:0032259,Particulate Matter results in decreased methyl...,Homo sapiens,Cells,ADGRV1|AS3MT|ATG13|BIN3|CCDC116|CMTR2|COMT|COQ...,1,22055874,candidate_contained_in_query,0.9,human_only


In [13]:
# Vérifier que outcome_key existe
print("outcome_key in main:", "outcome_key" in main.columns)
print("outcome_key in disease_mapping:", "outcome_key" in disease_mapping.columns)

# Sélectionner seulement les colonnes utiles du mapping pour éviter les doublons
disease_mapping_small = disease_mapping[
    [
        "outcome_key",
        # "ctd_mapping_decision",
        # "ctd_mapping_reason",
        "ctd_pm_disease_candidate_match",
        "ctd_pm_disease_match",
        "ctd_disease_name",
        "ctd_disease_id",
        "ctd_disease_match_type",
        "ctd_disease_match_confidence",
    ]
].copy()

# Préfixer pour éviter conflit avec les colonnes déjà présentes dans main
disease_mapping_small = disease_mapping_small.rename(
    columns={
        col: f"disease_map_{col}"
        for col in disease_mapping_small.columns
        if col != "outcome_key"
    }
)

main_with_disease_mapping = main.merge(
    disease_mapping_small,
    on="outcome_key",
    how="left",
    validate="m:1"
)

print("main:", main.shape)
print("main_with_disease_mapping:", main_with_disease_mapping.shape)

assert len(main_with_disease_mapping) == len(main)

main_with_disease_mapping.to_csv(OUT / "main_with_disease_mapping_qc.csv", index=False)

main_with_disease_mapping.head()

outcome_key in main: True
outcome_key in disease_mapping: True
main: (2368, 70)
main_with_disease_mapping: (2368, 76)


,association_id,pmid,association_label,association_claim,pollutant_outcome_key,pollutant_general,pollutant_specific,pollutant_original_mention,outcome_normalized,outcome_core,...,outcome_specific_ctd_match_available,pm_ctd_gene_resources_available,pm_ctd_phenotype_resources_available,mechanism_available,disease_map_ctd_pm_disease_candidate_match,disease_map_ctd_pm_disease_match,disease_map_ctd_disease_name,disease_map_ctd_disease_id,disease_map_ctd_disease_match_type,disease_map_ctd_disease_match_confidence
0,assoc_12842775_fe9cb68c6a56,12842775,PM10 -> daily mortality,PM10 is associated with increased daily mortality,pm10__daily_mortality,PM,PM10,PM10,daily mortality,daily mortality,...,False,True,True,True,False,False,Infertility,MESH:D007246,below_threshold,0.5385
1,assoc_12882940_64b5dfa384e6,12882940,NO2 -> persistent cough,NO2 is associated with increased persistent cough,no2__persistent_cough,NO2,NO2,nitrogen dioxide,persistent cough,persistent cough,...,False,False,False,False,True,True,Cough,MESH:D003371,candidate_contained_in_query,0.9000
2,assoc_14998750_49915f02c18a,14998750,PM2.5 -> decline in standard deviation of all ...,PM2.5 is associated with increased decline in ...,pm2_5__decline_in_standard_deviation_of_all_no...,PM,PM2.5,fine particulate matter (PM2.5),decline in standard deviation of all normal R-...,decline in standard deviation of all normal r ...,...,False,True,True,True,False,False,CRANIOSYNOSTOSIS AND DENTAL ANOMALIES,OMIM:614188,below_threshold,0.4681
3,assoc_14998750_7c8fc6671cfd,14998750,PM2.5 -> decline in short-term heart rate vari...,PM2.5 is associated with increased decline in ...,pm2_5__decline_in_short_term_heart_rate_variab...,PM,PM2.5,fine particulate matter (PM2.5),decline in short-term heart rate variability,decline in short term heart rate variability,...,False,True,True,True,False,False,X-Linked Intellectual Disability,MESH:D038901,below_threshold,0.4737
4,assoc_14998750_44ce04994750,14998750,PM2.5 -> C-reactive protein (CRP) increase,PM2.5 is associated with increased C-reactive ...,pm2_5__c_reactive_protein_crp_increase,PM,PM2.5,fine particulate matter (PM2.5),C-reactive protein (CRP) increase,c reactive protein (crp) increase,...,False,True,True,True,False,False,Activated Protein C Resistance,MESH:D020016,below_threshold,0.6885


In [15]:
phenotype_mapping_small = phenotype_mapping[
    [
        "outcome_key",
        # "ctd_mapping_decision",
        # "ctd_mapping_reason",
        "ctd_pm_phenotype_candidate_match",
        "ctd_pm_phenotype_match",
        "ctd_phenotype_name",
        "ctd_phenotype_id",
        "ctd_phenotype_match_type",
        "ctd_phenotype_match_confidence",
    ]
].copy()

phenotype_mapping_small = phenotype_mapping_small.rename(
    columns={
        col: f"phenotype_map_{col}"
        for col in phenotype_mapping_small.columns
        if col != "outcome_key"
    }
)

main_with_phenotype_mapping = main.merge(
    phenotype_mapping_small,
    on="outcome_key",
    how="left",
    validate="m:1"
)

print("main:", main.shape)
print("main_with_phenotype_mapping:", main_with_phenotype_mapping.shape)

assert len(main_with_phenotype_mapping) == len(main)

main_with_phenotype_mapping.to_csv(OUT / "main_with_phenotype_mapping_qc.csv", index=False)

main_with_phenotype_mapping.head()

main: (2368, 70)
main_with_phenotype_mapping: (2368, 76)


,association_id,pmid,association_label,association_claim,pollutant_outcome_key,pollutant_general,pollutant_specific,pollutant_original_mention,outcome_normalized,outcome_core,...,outcome_specific_ctd_match_available,pm_ctd_gene_resources_available,pm_ctd_phenotype_resources_available,mechanism_available,phenotype_map_ctd_pm_phenotype_candidate_match,phenotype_map_ctd_pm_phenotype_match,phenotype_map_ctd_phenotype_name,phenotype_map_ctd_phenotype_id,phenotype_map_ctd_phenotype_match_type,phenotype_map_ctd_phenotype_match_confidence
0,assoc_12842775_fe9cb68c6a56,12842775,PM10 -> daily mortality,PM10 is associated with increased daily mortality,pm10__daily_mortality,PM,PM10,PM10,daily mortality,daily mortality,...,False,True,True,True,False,False,cell chemotaxis,GO:0060326,below_threshold,0.4667
1,assoc_12882940_64b5dfa384e6,12882940,NO2 -> persistent cough,NO2 is associated with increased persistent cough,no2__persistent_cough,NO2,NO2,nitrogen dioxide,persistent cough,persistent cough,...,False,False,False,False,False,False,import into cell,GO:0098657,below_threshold,0.4375
2,assoc_14998750_49915f02c18a,14998750,PM2.5 -> decline in standard deviation of all ...,PM2.5 is associated with increased decline in ...,pm2_5__decline_in_standard_deviation_of_all_no...,PM,PM2.5,fine particulate matter (PM2.5),decline in standard deviation of all normal R-...,decline in standard deviation of all normal r ...,...,False,True,True,True,False,False,negative regulation of telomere maintenance,GO:0032205,below_threshold,0.4600
3,assoc_14998750_7c8fc6671cfd,14998750,PM2.5 -> decline in short-term heart rate vari...,PM2.5 is associated with increased decline in ...,pm2_5__decline_in_short_term_heart_rate_variab...,PM,PM2.5,fine particulate matter (PM2.5),decline in short-term heart rate variability,decline in short term heart rate variability,...,False,True,True,True,False,False,regulation of heart rate,GO:0002027,below_threshold,0.5000
4,assoc_14998750_44ce04994750,14998750,PM2.5 -> C-reactive protein (CRP) increase,PM2.5 is associated with increased C-reactive ...,pm2_5__c_reactive_protein_crp_increase,PM,PM2.5,fine particulate matter (PM2.5),C-reactive protein (CRP) increase,c reactive protein (crp) increase,...,False,True,True,True,False,False,activation of protein kinase B activity,GO:0032148,below_threshold,0.5429


In [16]:
main_complete_flat = (
    main
    .merge(disease_mapping_small, on="outcome_key", how="left", validate="m:1")
    .merge(phenotype_mapping_small, on="outcome_key", how="left", validate="m:1")
    .merge(gene_agg, on="association_id", how="left", validate="1:1")
    .merge(phenotype_agg, on="association_id", how="left", validate="1:1")
)

for col in ["gene_count", "phenotype_count"]:
    if col in main_complete_flat.columns:
        main_complete_flat[col] = main_complete_flat[col].fillna(0).astype(int)

print("main:", main.shape)
print("main_complete_flat:", main_complete_flat.shape)

assert len(main_complete_flat) == len(main), "Erreur: la fusion complète plate a changé le nombre de lignes."

main_complete_flat.to_csv(OUT / "main_complete_flat_all_summary.csv", index=False)

main_complete_flat.head()

main: (2368, 70)
main_complete_flat: (2368, 91)


,association_id,pmid,association_label,association_claim,pollutant_outcome_key,pollutant_general,pollutant_specific,pollutant_original_mention,outcome_normalized,outcome_core,...,phenotype_map_ctd_phenotype_match_confidence,gene_count,gene_symbols,gene_ids,gene_actions,gene_reference_count_sum,phenotype_count,phenotype_names,phenotype_ids,phenotype_interactions
0,assoc_12842775_fe9cb68c6a56,12842775,PM10 -> daily mortality,PM10 is associated with increased daily mortality,pm10__daily_mortality,PM,PM10,PM10,daily mortality,daily mortality,...,0.4667,200,ADRB2;AHR;AKR1C1;AKR1C3;AKT1;ALB;ALDH1A3;ALDH3...,1000;100506658;102466659;1026;1029;10397;1052;...,affects^cotreatment;decreases^expression;incre...,2716.0,0,NaN,NaN,NaN
1,assoc_12882940_64b5dfa384e6,12882940,NO2 -> persistent cough,NO2 is associated with increased persistent cough,no2__persistent_cough,NO2,NO2,nitrogen dioxide,persistent cough,persistent cough,...,0.4375,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
2,assoc_14998750_49915f02c18a,14998750,PM2.5 -> decline in standard deviation of all ...,PM2.5 is associated with increased decline in ...,pm2_5__decline_in_standard_deviation_of_all_no...,PM,PM2.5,fine particulate matter (PM2.5),decline in standard deviation of all normal R-...,decline in standard deviation of all normal r ...,...,0.4600,200,ADRB2;AHR;AKR1C1;AKR1C3;AKT1;ALB;ALDH1A3;ALDH3...,1000;100506658;102466659;1026;1029;10397;1052;...,affects^cotreatment;decreases^expression;incre...,2716.0,0,NaN,NaN,NaN
3,assoc_14998750_7c8fc6671cfd,14998750,PM2.5 -> decline in short-term heart rate vari...,PM2.5 is associated with increased decline in ...,pm2_5__decline_in_short_term_heart_rate_variab...,PM,PM2.5,fine particulate matter (PM2.5),decline in short-term heart rate variability,decline in short term heart rate variability,...,0.5000,200,ADRB2;AHR;AKR1C1;AKR1C3;AKT1;ALB;ALDH1A3;ALDH3...,1000;100506658;102466659;1026;1029;10397;1052;...,affects^cotreatment;decreases^expression;incre...,2716.0,0,NaN,NaN,NaN
4,assoc_14998750_44ce04994750,14998750,PM2.5 -> C-reactive protein (CRP) increase,PM2.5 is associated with increased C-reactive ...,pm2_5__c_reactive_protein_crp_increase,PM,PM2.5,fine particulate matter (PM2.5),C-reactive protein (CRP) increase,c reactive protein (crp) increase,...,0.5429,200,ADRB2;AHR;AKR1C1;AKR1C3;AKT1;ALB;ALDH1A3;ALDH3...,1000;100506658;102466659;1026;1029;10397;1052;...,affects^cotreatment;decreases^expression;incre...,2716.0,0,NaN,NaN,NaN


In [17]:
main_complete_flat.columns.to_list()

['association_id',
 'pmid',
 'association_label',
 'association_claim',
 'pollutant_outcome_key',
 'pollutant_general',
 'pollutant_specific',
 'pollutant_original_mention',
 'outcome_normalized',
 'outcome_core',
 'outcome_category',
 'population_group',
 'is_human_study',
 'measure_type',
 'effect_value',
 'effect_value_numeric',
 'effect_direction',
 'ci_level',
 'ci_lower',
 'ci_upper',
 'p_value',
 'statistical_significance',
 'effect_summary',
 'exposure_increment',
 'exposure_increment_value',
 'exposure_increment_unit',
 'adjustment_status',
 'analysis_method',
 'qc_result_context',
 'evidence_sentence',
 'is_pm_d052638_candidate',
 'ctd_chemical_id',
 'ctd_chemical_name',
 'ctd_pollutant_match_type',
 'ctd_pollutant_match_confidence',
 'mechanistic_enrichment_scope',
 'outcome_query',
 'outcome_key',
 'ctd_pm_disease_candidate_match',
 'ctd_pm_disease_match',
 'ctd_disease_mapping_decision',
 'ctd_disease_mapping_reason',
 'ctd_disease_name',
 'ctd_disease_id',
 'ctd_disease_d

In [18]:
keep_cols = [
    # Identifiants
    "association_id",
    "pmid",

    # Association
    "association_label",
    "association_claim",
    "pollutant_outcome_key",

    # Polluant
    "pollutant_general",
    "pollutant_specific",
    "pollutant_original_mention",

    # Outcome
    "outcome_normalized",
    "outcome_core",
    "outcome_category",
    "population_group",
    "is_human_study",

    # Effet statistique
    "measure_type",
    "effect_value",
    "effect_value_numeric",
    "effect_direction",
    "ci_level",
    "ci_lower",
    "ci_upper",
    "p_value",
    "statistical_significance",
    "effect_summary",

    # Exposition
    "exposure_increment",
    "exposure_increment_value",
    "exposure_increment_unit",

    # Méthode / contexte scientifique
    "adjustment_status",
    "analysis_method",
    "qc_result_context",

    # Preuve textuelle
    "evidence_sentence",

    # CTD pollutant / PM enrichment
    "is_pm_d052638_candidate",
    "ctd_chemical_id",
    "ctd_chemical_name",
    "ctd_pollutant_match_type",
    "mechanistic_enrichment_scope",

    # CTD disease accepté
    "ctd_pm_disease_match",
    "ctd_disease_name",
    "ctd_disease_id",
    "ctd_disease_direct_evidence",
    "ctd_disease_inference_score",
    "ctd_disease_reference_count",

    # CTD phenotype accepté
    "ctd_pm_phenotype_match",
    "ctd_phenotype_name",
    "ctd_phenotype_id",
    "ctd_phenotype_interaction",
    "ctd_phenotype_reference_count",

    # Statut enrichissement
    "outcome_mapping_type",
    "outcome_specific_ctd_match_available",
    "mechanism_available",

    # Genes résumé
    "linked_human_gene_total_available",
    "gene_count",
    "gene_symbols",
    "gene_ids",
    "gene_actions",
    "gene_reference_count_sum",

    # Phenotypes résumé
    "linked_human_phenotype_total_available",
    "phenotype_count",
    "phenotype_names",
    "phenotype_ids",
    "phenotype_interactions",
]

In [19]:
output_path = Path("data/processed_full/manual_fusions/ctd_extraction_final.csv")

final_cols = [c for c in keep_cols if c in main_complete_flat.columns]

clean_df = main_complete_flat[final_cols].copy()

print("Input:",  main_complete_flat.shape)
print("Output:", clean_df.shape)

clean_df.to_csv(output_path, index=False)

print("Saved:", output_path)

Input: (2368, 91)
Output: (2368, 60)
Saved: data/processed_full/manual_fusions/ctd_extraction_final.csv


#### Merge to obtain full dataset contains genees and phenotypes

In [4]:

extract_path = OUT / "ctd_extraction_final.csv"
genes_path = OUT / "main_gene_meta_long.csv"
phenotypes_path = OUT / "main_phenotype_long.csv"

output_path = OUT / "ctd_extraction_final_full.csv"

In [5]:
extract = pd.read_csv(extract_path, low_memory=False)
genes = pd.read_csv(genes_path, low_memory=False)
phenotypes = pd.read_csv(phenotypes_path, low_memory=False)

In [6]:

extract["association_id"] = extract["association_id"].astype(str)
genes["association_id"] = genes["association_id"].astype(str)
phenotypes["association_id"] = phenotypes["association_id"].astype(str)

In [8]:
# Garder uniquement les colonnes gene non déjà présentes dans main
gene_cols = ["association_id"] + [
    col for col in genes.columns
    if col not in extract.columns and col != "association_id"
]

genes_clean = genes[gene_cols].copy()

In [9]:
# Garder uniquement les colonnes phenotype non déjà présentes dans main
phenotype_cols = ["association_id"] + [
    col for col in phenotypes.columns
    if col not in extract.columns and col != "association_id"
]

phenotypes_clean = phenotypes[phenotype_cols].copy()

In [10]:
# Fusion main + genes
merged = extract.merge(
    genes_clean,
    on="association_id",
    how="left"
)


: 

In [ ]:
# Fusion résultat + phenotypes
merged = merged.merge(
    phenotypes_clean,
    on="association_id",
    how="left"
)

In [ ]:
print("genes_clean:", genes_clean.shape)
print("phenotypes_clean:", phenotypes_clean.shape)
print("merged:", merged.shape)

merged.to_csv(output_path, index=False)

print("Saved:", output_path)

In [4]:
final_data = pd.read_csv('./ctd_extraction_final.csv', low_memory=False)
final_data.head(5)

,association_id,pmid,pollutant_general,pollutant_specific,outcome_normalized,outcome_category,measure_type,effect_value_numeric,effect_direction,exposure_increment_value,...,gene_id,gene_interaction_actions_top,population_group,age_group,age_min,age_max,sex,study_country,study_population,evidence_sentence
0,assoc_12842775_fe9cb68c6a56,12842775,PM,PM10,daily mortality,general_mortality,percent_change,0.5,increase,10.0,...,3576.0,increases^expression;decreases^reaction;increa...,study subjects,not_reported,NaN,NaN,not_reported,United Kingdom,persons known to have died from the December 1...,A 10-µg/m³ increase in PM10 has been found to ...
1,assoc_12842775_fe9cb68c6a56,12842775,PM,PM10,daily mortality,general_mortality,percent_change,0.5,increase,10.0,...,3569.0,increases^expression;decreases^reaction;increa...,study subjects,not_reported,NaN,NaN,not_reported,United Kingdom,persons known to have died from the December 1...,A 10-µg/m³ increase in PM10 has been found to ...
2,assoc_12842775_fe9cb68c6a56,12842775,PM,PM10,daily mortality,general_mortality,percent_change,0.5,increase,10.0,...,7124.0,decreases^reaction;increases^expression;increa...,study subjects,not_reported,NaN,NaN,not_reported,United Kingdom,persons known to have died from the December 1...,A 10-µg/m³ increase in PM10 has been found to ...
3,assoc_12842775_fe9cb68c6a56,12842775,PM,PM10,daily mortality,general_mortality,percent_change,0.5,increase,10.0,...,3553.0,increases^expression;decreases^reaction;increa...,study subjects,not_reported,NaN,NaN,not_reported,United Kingdom,persons known to have died from the December 1...,A 10-µg/m³ increase in PM10 has been found to ...
4,assoc_12842775_fe9cb68c6a56,12842775,PM,PM10,daily mortality,general_mortality,percent_change,0.5,increase,10.0,...,5970.0,decreases^reaction;increases^expression;increa...,study subjects,not_reported,NaN,NaN,not_reported,United Kingdom,persons known to have died from the December 1...,A 10-µg/m³ increase in PM10 has been found to ...


In [3]:
time_data = pd.read_csv('./data/processed_full/exposure_temporal_extraction/temp_exposure_time.csv', low_memory=False)
time_data.head(5)

,association_id,pmid,exposure_window_category,exposure_window_text,exposure_duration_value,exposure_duration_unit,exposure_lag,exposure_period,exposure_averaging_window,exposure_assessment_method,temporal_evidence_sentence,temporal_source_section,temporal_confidence,temporal_status
0,assoc_12842775_fe9cb68c6a56,12842775,short_term,daily,NaN,day,not_reported,not_reported,not_reported,not_reported,A 10- micro g m(-3) increase in PM10 (PM < 10 ...,ARTICLE_TEMPORAL_CONTEXT,high,explicit
1,assoc_12882940_64b5dfa384e6,12882940,early_life,first year of life,1.0,year,not_reported,not_reported,not_reported,not_reported,the authors examined the relations of exposure...,PUBMED_ABSTRACT,high,explicit
2,assoc_14998750_49915f02c18a,14998750,short_term,short-term variability in temporal deviations ...,NaN,not_reported,not_reported,not_reported,not_reported,regression analysis,Most short-term variability in temporal deviat...,ARTICLE_TEMPORAL_CONTEXT,medium,explicit
3,assoc_14998750_7c8fc6671cfd,14998750,not_reported,not_reported,NaN,not_reported,not_reported,not_reported,not_reported,not_reported,not_reported,[ARTICLE_TEMPORAL_CONTEXT],low,not_reported
4,assoc_14998750_44ce04994750,14998750,short_term,short-term variability,NaN,not_reported,not_reported,not_reported,not_reported,not_reported,Most short-term variability in temporal deviat...,[ARTICLE_TEMPORAL_CONTEXT] (last paragraph),high,explicit


In [5]:
final_data["association_id"] = final_data["association_id"].astype(str)
time_data["association_id"] = time_data["association_id"].astype(str)

final_data['pmid'] = final_data['pmid'].astype(str)
time_data['pmid'] = time_data['pmid'].astype(str)


time_data = time_data.drop_duplicates(
    subset=["association_id", "pmid"],
    keep="last"
)

In [7]:
merged_data_time = final_data.merge(
    time_data,
    on=["association_id", "pmid"],
    how="left",
    validate="m:1"
)

In [8]:
print("Final avant merge:", final_data.shape)
print("Temporalité:", time_data.shape)
print("Final après merge:", merged_data_time.shape)

Final avant merge: (98688, 31)
Temporalité: (2368, 14)
Final après merge: (98688, 43)


In [9]:
merged_data_time.to_csv('./ctd_extraction_temporal.csv', index=False)

In [10]:
merged_data_time.head(5)

,association_id,pmid,pollutant_general,pollutant_specific,outcome_normalized,outcome_category,measure_type,effect_value_numeric,effect_direction,exposure_increment_value,...,exposure_duration_value,exposure_duration_unit,exposure_lag,exposure_period,exposure_averaging_window,exposure_assessment_method,temporal_evidence_sentence,temporal_source_section,temporal_confidence,temporal_status
0,assoc_12842775_fe9cb68c6a56,12842775,PM,PM10,daily mortality,general_mortality,percent_change,0.5,increase,10.0,...,NaN,day,not_reported,not_reported,not_reported,not_reported,A 10- micro g m(-3) increase in PM10 (PM < 10 ...,ARTICLE_TEMPORAL_CONTEXT,high,explicit
1,assoc_12842775_fe9cb68c6a56,12842775,PM,PM10,daily mortality,general_mortality,percent_change,0.5,increase,10.0,...,NaN,day,not_reported,not_reported,not_reported,not_reported,A 10- micro g m(-3) increase in PM10 (PM < 10 ...,ARTICLE_TEMPORAL_CONTEXT,high,explicit
2,assoc_12842775_fe9cb68c6a56,12842775,PM,PM10,daily mortality,general_mortality,percent_change,0.5,increase,10.0,...,NaN,day,not_reported,not_reported,not_reported,not_reported,A 10- micro g m(-3) increase in PM10 (PM < 10 ...,ARTICLE_TEMPORAL_CONTEXT,high,explicit
3,assoc_12842775_fe9cb68c6a56,12842775,PM,PM10,daily mortality,general_mortality,percent_change,0.5,increase,10.0,...,NaN,day,not_reported,not_reported,not_reported,not_reported,A 10- micro g m(-3) increase in PM10 (PM < 10 ...,ARTICLE_TEMPORAL_CONTEXT,high,explicit
4,assoc_12842775_fe9cb68c6a56,12842775,PM,PM10,daily mortality,general_mortality,percent_change,0.5,increase,10.0,...,NaN,day,not_reported,not_reported,not_reported,not_reported,A 10- micro g m(-3) increase in PM10 (PM < 10 ...,ARTICLE_TEMPORAL_CONTEXT,high,explicit


In [ ]:
# filter pour garder toutes les colonnes sauf temporal_evidence_sentence, temporal_status
merged_data_time = merged_data_time.drop(columns=['temporal_evidence_sentence', 'temporal_status', 'exposure_window_text', 'exposure_duration_value', 'exposure_period', 'exposure_averaging_window', 'exposure_assessment_method', 'temporal_source_section', 'temporal_confidence'])

In [12]:
merged_data_time.to_csv('./ctd_extraction_temporal.csv', index=False)